# OpenAI Batch Runs


## Setup


In [ ]:
# Libraries
import json
import sys
import time
from pathlib import Path

import pandas as pd

In [ ]:
# Project Root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Pipeline
%load_ext autoreload
%autoreload 2

import backends
import run
import settings
import utils

# Notebook Calls Functions
needs = {'run': ['write_batch', 'read_batch', 'batch_path', 'name_after_job',
                 'set_aside_replies'],
         'utils': ['api_key', 'read_lines', 'read_table', 'result_path',
                   'model_slug', 'make_directories'],
         'backends': ['USAGE', 'spent', 'record_usage'],
         'settings': ['BATCHES_DIR', 'MODELS', 'GENERATION']}
missing = [f'{name}.{attr}' for name, attrs in needs.items()
           for attr in attrs if not hasattr(globals()[name], attr)]
if missing:
    raise SystemExit('Scripts are out of date, missing: ' + ', '.join(missing)
                     + '\nCopy scripts/ from the latest package and restart the kernel.')

# Pipeline Check
probe = backends.build_payload('openai', 'probe', [{'role': 'user', 'content': 'x'}],
                               settings.GENERATION['max_tokens'],
                               settings.GENERATION['temperature'])
if 'temperature' not in probe or 'top_p' not in probe:
    raise SystemExit('backends.py does not send sampling parameters to OpenAI, so '
                     'this arm would run at provider defaults.\nCopy scripts/ from '
                     'the latest package and restart the kernel.')

utils.make_directories()
pd.set_option('display.max_colwidth', 70)

## Model


In [ ]:
MODEL = 'gpt-5.6-luna'
ENDPOINT = '/v1/responses'

spec = next(e for e in settings.MODELS.values() if e['id'] == MODEL)
if spec['provider'] != 'openai':
    raise SystemExit(f'{MODEL} is served by {spec["provider"]}, not OpenAI')

prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
have = len(utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR)))

print("Model:", MODEL)
print("Endpoint:", ENDPOINT)
print("Pricing:", spec['price']['input'], spec['price']['output'])
print("Temperature:", 'as the design asks' if backends.takes_sampling(MODEL) else 'provider defaults', settings.GENERATION['temperature'])
print("Reasoning:", spec.get('reasoning') or 'provider default')
print("Token Cap:", settings.GENERATION['max_tokens'])
print("Collected:", have, wanted)
print("API Key:", bool(utils.api_key('openai')))

## Existing Results


In [ ]:
# Existing Results
rows = []
for kind, folder in [('collected', settings.ADAPTATION_DIR),
                     ('superseded', settings.ADAPTATION_DIR.parent / 'superseded')]:
    for file in sorted(folder.glob(f'{utils.model_slug(MODEL)}*.jsonl')):
        rows.append({'where': kind, 'file': file.name,
                     'replies': len(utils.read_lines(file)), 'sampling': ''})

for file in sorted(settings.BATCHES_DIR.glob('*_requests.jsonl')):
    body = json.loads(file.read_text().splitlines()[0])['body']
    if body.get('model') != MODEL:
        continue
    rows.append({'where': 'request file', 'file': file.name,
                 'replies': sum(1 for _ in file.open()),
                 'sampling': f"temp {body.get('temperature', 'provider default')}, "
                             f"top_p {body.get('top_p', 'provider default')}, "
                             f"reasoning {body.get('reasoning', {}).get('effort', 'default')}"})

display(pd.DataFrame(rows) if rows else 'Nothing on disk for this model yet')

## Rerun


In [ ]:
FRESH = False        # Fresh Run

if FRESH:
    moved = run.set_aside_replies(MODEL)
    print("Collected:", moved)
else:
    print("Resume Mode:")

## Requests


In [ ]:
path, count = run.write_batch(MODEL, endpoint=ENDPOINT)

if path is None:
    print("Model:")
else:
    print("Requests Written:", count, path)
    print()
    print("Request:", json.dumps(json.loads(path.read_text().splitlines()[0]), indent=2))

## Cost


In [ ]:
OUTPUT_TOKENS = 211          # Output Tokens
INPUT_TOKENS = 24            # Output Tokens

if path is None:
    print("Model:")
else:
    price = spec['price']
    standard = (count * INPUT_TOKENS * price['input']
                + count * OUTPUT_TOKENS * price['output']) / 1e6
    print("Calls:", count, OUTPUT_TOKENS)
    print("Standard Cost:", standard)
    print("Batch Cost:", standard / 2)

## Submit


In [ ]:
PROVIDER = backends.provider_of(MODEL)

if PROVIDER == 'openai':
    from openai import OpenAI
    client = OpenAI(api_key=utils.api_key('openai'))
    uploaded = client.files.create(file=open(path, 'rb'), purpose='batch')
    job = client.batches.create(input_file_id=uploaded.id, endpoint=ENDPOINT,
                                completion_window='24h')
    job_id, status = job.id, job.status
elif PROVIDER == 'anthropic':
    import anthropic
    client = anthropic.Anthropic(api_key=utils.api_key('anthropic'))
    # Inline Requests
    requests = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    job = client.messages.batches.create(requests=requests)
    job_id, status = job.id, job.processing_status
else:
    raise SystemExit(f'{PROVIDER} batches are not driven from this notebook. '
                     f'Export the file and use the provider console.')

# Job Record
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job_file.parent.mkdir(parents=True, exist_ok=True)
job_file.write_text(job_id)
print("Submitted:", job_id, status)

# Request Record
print("Request File:", run.name_after_job(MODEL, job_id))

## Resume Job


In [ ]:
# OpenAI Jobs
if PROVIDER != 'openai':
    print("Provider Note:", PROVIDER)
    jobs = []
else:
    jobs = [{'id': b.id, 'status': b.status, 'endpoint': b.endpoint,
             'total': b.request_counts.total,
             'completed': b.request_counts.completed,
             'failed': b.request_counts.failed,
             'created': pd.to_datetime(b.created_at, unit='s')}
            for b in client.batches.list(limit=10).data]
    display(pd.DataFrame(jobs))

In [ ]:
# Select Job
ADOPT = jobs[0]['id'] if 'jobs' in dir() and jobs else ''

if ADOPT:
    job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
    job_file.parent.mkdir(parents=True, exist_ok=True)
    job_file.write_text(ADOPT)
    print("Job Adopted:", ADOPT, MODEL)
    print("Status Ready:")

## Status


In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')

if not job_file.exists():
    print("Model:")
elif PROVIDER == 'openai':
    job = client.batches.retrieve(job_file.read_text().strip())
    done, failed = job.request_counts.completed, job.request_counts.failed
    total = job.request_counts.total or 1
    print("Job ID:", job.id)
    print("Job Status:", job.status.capitalize(), done)
else:
    job = client.messages.batches.retrieve(job_file.read_text().strip())
    counts = job.request_counts
    done = counts.succeeded + counts.errored + counts.canceled + counts.expired
    total = done + counts.processing or 1
    print("Job ID:", job.id)
    print("Job Status:", job.processing_status.capitalize(), counts.succeeded)

## Results


In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job_id = job_file.read_text().strip() if job_file.exists() else None

if job_id is None:
    print("Submitted:")
else:
    results = run.batch_path(MODEL, 'output', job_id)
    if PROVIDER == 'openai':
        job = client.batches.retrieve(job_id)
        ready = job.status == 'completed'
        if ready:
            results.write_bytes(client.files.content(job.output_file_id).read())
    else:
        job = client.messages.batches.retrieve(job_id)
        ready = job.processing_status == 'ended'
        if ready:
            with results.open('w') as file:
                for entry in client.messages.batches.results(job_id):
                    file.write(entry.to_json() + '\n')

    if not ready:
        print("Pending:")
    else:
        print("Downloaded:", results)
        first = json.loads(results.read_text().splitlines()[0])
        print("First ID:", first['custom_id'])

        backends.USAGE.update(calls=0, input=0, output=0)
        read, failed, truncated, repeated, blocked = run.read_batch(MODEL, results)

        usage, cost = backends.USAGE, backends.spent(MODEL)
        print("Replies Read:", read, failed)
        print("Token Usage:", usage['input'], usage['output'])
        print("Standard Cost:", cost, cost / 2)
        print("Mean Tokens:", usage['output'] / max(read - failed, 1), OUTPUT_TOKENS)

## Validation


In [ ]:
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print("Collected:", MODEL)
else:
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print("Replies:", len(collected), blank)
    print("Coverage:", collected['prompt_id'].nunique(), len(prompts))

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))